# Phase 5 — Human Review and Audit Trail

## Overview

Phase 5 introduced a human-in-the-loop review workflow on top of the validation and anomaly detection completed in Phase 4.

The main goals were to:

* decide which documents can be automatically accepted,
* route uncertain or problematic documents for human review,
* support human approval, rejection, and correction,
* preserve machine decisions and human actions,
* maintain an append-only audit history,
* verify the complete review workflow end to end.

Phase 5 was divided into:

* **Phase 5A — Review Decision Engine**
* **Phase 5B — Human Review / Override**
* **Phase 5C — Audit Trail**
* **Phase 5D — End-to-End Review Workflow**

---

# Phase 5A — Review Decision Engine

## Objective

The review decision engine converts Phase 4 anomaly results into an operational review decision.

The system intentionally uses only two machine-level outcomes:

* **AUTO_ACCEPT**
* **REVIEW_REQUIRED**

Automatic rejection was not introduced. Documents containing errors or uncertainty are escalated to a human reviewer instead.

---

## Review Priority Logic

Review cases were assigned priorities according to anomaly severity.

| Condition                                 | Decision        | Priority |
| ----------------------------------------- | --------------- | -------- |
| No errors or warnings                     | AUTO_ACCEPT     | NONE     |
| Expiring soon only                        | REVIEW_REQUIRED | LOW      |
| Warning such as expired or low confidence | REVIEW_REQUIRED | MEDIUM   |
| Any blocking error                        | REVIEW_REQUIRED | HIGH     |

Examples of high-priority conditions included:

* missing critical field,
* unknown document type,
* invalid critical evidence,
* logical date errors.

Examples of medium-priority conditions included:

* expired document,
* low-confidence critical field,
* invalid optional-field evidence.

---

## Deterministic Tests

The review decision engine was tested with:

* clean document,
* expiring-soon document,
* expired document,
* invalid optional field,
* low critical-field confidence,
* missing critical field,
* unknown document type,
* combined warning and error conditions.

All tests passed.

A combined warning and error correctly escalated to **HIGH** priority.

---

## Real-Document Integration Tests

### Guard Licence

The document was structurally valid but expired.

Result:

* Decision: **REVIEW_REQUIRED**
* Priority: **MEDIUM**
* Reason: **DOCUMENT_EXPIRED**

### SIA Badge

One run missed the critical full-name field while the document was also expired.

Result:

* Decision: **REVIEW_REQUIRED**
* Priority: **HIGH**
* Reasons:

  * missing critical field,
  * expired document.

This demonstrated correct escalation when an error and warning coexist.

### ID Card

The date of birth was extracted but lacked reliable semantic evidence.

Result:

* Decision: **REVIEW_REQUIRED**
* Priority: **MEDIUM**
* Reason: invalid evidence for an extracted non-critical field.

---

# Phase 5B — Human Review and Override

## Objective

Phase 5B introduced explicit human actions after machine review.

Supported actions were:

* **APPROVE**
* **REJECT**
* **CORRECT**

The machine decision is preserved separately from the human decision.

---

## Approve

A reviewer may approve a document even when the machine requested review.

This supports cases where the document is valid after manual inspection despite a warning.

---

## Reject

A reviewer may reject a document after inspecting the machine findings and source document.

This provides a final human decision without allowing the automated pipeline to make irreversible rejection decisions.

---

## Correct

A reviewer may provide corrected values for supported fields.

The correction does not silently overwrite the original machine extraction.

Instead, both are preserved:

* original machine result,
* human correction.

This enables later comparison and traceability.

---

## Validation of Human Actions

The workflow also prevents invalid review operations.

Tests confirmed that:

* `CORRECT` requires at least one correction,
* corrections cannot be attached to `APPROVE` or `REJECT`,
* unsupported correction fields are rejected.

All human-review tests passed.

---

# Phase 5C — Audit Trail

## Objective

Phase 5C introduced an append-only audit trail to record both machine and human activity.

A JSON Lines audit log was used during this phase as lightweight persistent storage before database integration in Phase 6.

---

## Machine Audit Events

Machine review decisions were recorded with information such as:

* document identifier,
* machine decision,
* review-required state,
* priority,
* reason codes,
* timestamp.

This preserves why a document entered review.

---

## Human Audit Events

Human review events preserved:

* reviewer identifier,
* human action,
* associated machine decision,
* machine priority,
* original machine reason codes,
* reviewer notes,
* corrections,
* review timestamp.

---

## Correction History

Human corrections were stored as part of the audit event rather than replacing the machine-generated values.

This established a traceable sequence:

**Machine extraction → Machine review decision → Human correction**

The original state therefore remains available for later evaluation and investigation.

---

## Audit Tests

The audit layer successfully tested:

* machine decision logging,
* human review logging,
* reviewer identity preservation,
* document history retrieval,
* correction storage,
* append-only behavior,
* history separation between different documents.

All audit trail tests passed.

---

# Phase 5D — End-to-End Review Workflow

## Objective

Phase 5D verified that the Phase 5 components work correctly together.

The tested workflow was:

**Anomaly Result → Machine Review Decision → Human Action → Audit History**

---

## End-to-End Test Scenarios

### Clean Document

A clean document produced:

* AUTO_ACCEPT,
* no human review,
* one machine audit event.

**Result: Passed**

### Expired Document

An expired document produced:

* REVIEW_REQUIRED,
* MEDIUM priority,
* human APPROVE action,
* machine and human audit events.

**Result: Passed**

### Critical Error

A document with a missing critical field produced:

* REVIEW_REQUIRED,
* HIGH priority,
* human REJECT action,
* complete audit history.

**Result: Passed**

### Invalid Field Evidence

A document containing invalid evidence for an extracted field produced:

* REVIEW_REQUIRED,
* MEDIUM priority,
* human CORRECT action,
* correction preserved in the audit trail.

**Result: Passed**

---

## Machine Reason Preservation

The end-to-end tests verified that the original machine reason remains available after human review.

For example:

**EXTRACTED_FIELD_INVALID_EVIDENCE**

was preserved both:

* in the original machine event,
* in the later human-review record.

This prevents human action from erasing the reason the system originally requested review.

---

## Document History Isolation

Audit histories for multiple documents were tested independently.

Events from one document did not appear in another document's history.

This confirmed correct document-level audit separation.

---

# Key Findings

1. **Human review should be triggered by machine uncertainty rather than replaced by automatic rejection.**

2. **Errors and warnings require different operational treatment.** Errors produce high-priority review, while warnings may produce low or medium priority.

3. **Machine decisions and human decisions should remain separate.** This preserves accountability and enables later evaluation.

4. **Human corrections should not destroy original machine extraction results.**

5. **Reason codes must survive the complete workflow.** The original cause for review remains traceable after approval, rejection, or correction.

6. **Audit history should be append-only.** Existing events are preserved instead of overwritten.

7. **Per-document history isolation is essential** for reliable compliance and investigation workflows.

---

# Final Phase 5 Workflow

```text
Document Anomaly Result
        ↓
Review Decision Engine
        ↓
AUTO_ACCEPT
        OR
REVIEW_REQUIRED
        ↓
Human Reviewer
        ↓
APPROVE / REJECT / CORRECT
        ↓
Machine + Human Audit Events
        ↓
Complete Document Review History
```

---

# Final Conclusion

Phase 5 successfully added a complete human-in-the-loop review and traceability layer.

The system can now:

* automatically accept clean documents,
* escalate uncertain documents according to priority,
* support human approve, reject, and correct actions,
* preserve machine reason codes,
* retain correction history,
* record machine and human events,
* maintain append-only document histories,
* and execute the complete review workflow end to end.

**Phase 5A — Review Decision Engine: Complete**

**Phase 5B — Human Review / Override: Complete**

**Phase 5C — Audit Trail: Complete**

**Phase 5D — End-to-End Review Workflow: Complete**

**Phase 5: Complete**
